In [1]:
!pip install nltk scikit-learn

import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [2]:
from google.colab import files
upload=files.upload()

Saving archive (2).zip to archive (2).zip


In [5]:
import pandas as pd
import zipfile

zip_file_name = 'archive (2).zip'
csv_file_name_in_zip = 'imdb_list.csv' # Based on the content of the zip file in kernel state

with zipfile.ZipFile(zip_file_name, 'r') as z:
    with z.open(csv_file_name_in_zip) as f:
        df = pd.read_csv(f)

In [8]:
print(df.head())
print("\nShape:", df.shape)


   Unnamed: 0         id                               title  rating  \
0           0  tt0369610                      Jurassic World     6.9   
1           1  tt3774694                                Love     6.1   
2           2  tt2361509                          The Intern     7.1   
3           3  tt2381249  Mission: Impossible - Rogue Nation     7.4   
4           4  tt3460252                   The Hateful Eight     7.8   

                         genre  year  
0    Action, Adventure, Sci-Fi  2015  
1               Drama, Romance  2015  
2                Comedy, Drama  2015  
3  Action, Adventure, Thriller  2015  
4        Crime, Drama, Mystery  2015  

Shape: (250, 6)


**NLP Preprocessing**

In [11]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove special characters
    text = re.sub(r"[^a-z\s]", "", text)

    # Tokenize
    words = text.split()

    # Remove stopwords + Lemmatize
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)
df['clean_text'] = df['title'].apply(preprocess_text)

print(df[['title', 'clean_text']].head())

                                title                       clean_text
0                      Jurassic World                   jurassic world
1                                Love                             love
2                          The Intern                           intern
3  Mission: Impossible - Rogue Nation  mission impossible rogue nation
4                   The Hateful Eight                    hateful eight


**Feature Engineering**

In [12]:
bow = CountVectorizer(max_features=5000)

X_bow = bow.fit_transform(df['clean_text']).toarray()

In [13]:
tfidf = TfidfVectorizer(max_features=5000)

X_tfidf = tfidf.fit_transform(df['clean_text']).toarray()

**Train-Test Split**

In [15]:
df['label'] = (df['rating'] >= 7.0).astype(int)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

**Model Building**

1. Logistic Regression

In [16]:
lr = LogisticRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

2. Naive Bayes

In [17]:
nb = MultinomialNB()
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

3. Decision Tree

In [18]:
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

**Model Evaluation**

In [19]:
def evaluate_model(name, y_test, y_pred):
    print(f"\n===== {name} =====")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))


evaluate_model("Logistic Regression", y_test, y_pred_lr)
evaluate_model("Naive Bayes", y_test, y_pred_nb)
evaluate_model("Decision Tree", y_test, y_pred_dt)


===== Logistic Regression =====
Accuracy: 0.68
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        16
           1       0.68      1.00      0.81        34

    accuracy                           0.68        50
   macro avg       0.34      0.50      0.40        50
weighted avg       0.46      0.68      0.55        50


===== Naive Bayes =====
Accuracy: 0.7
              precision    recall  f1-score   support

           0       1.00      0.06      0.12        16
           1       0.69      1.00      0.82        34

    accuracy                           0.70        50
   macro avg       0.85      0.53      0.47        50
weighted avg       0.79      0.70      0.59        50


===== Decision Tree =====
Accuracy: 0.66
              precision    recall  f1-score   support

           0       0.33      0.06      0.11        16
           1       0.68      0.94      0.79        34

    accuracy                           0.66        50

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


**Comparison Table**

In [20]:
results = {
    "Model": ["Logistic Regression", "Naive Bayes", "Decision Tree"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_dt)
    ]
}

results_df = pd.DataFrame(results)
print(results_df)

                 Model  Accuracy
0  Logistic Regression      0.68
1          Naive Bayes      0.70
2        Decision Tree      0.66
